In [2]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install --upgrade datasets pyarrow

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
max_seq_length=1024
dtype=None
load_in_4bit=True

model,tokenizer=FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

In [ ]:
model=FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3047,
    use_rslora=False,
    loftq_config=None


    
)

In [ ]:
import pandas as pd
df=pd.read_csv("/kaggle/input/pubmed-article-summarization-dataset/train.csv")
df.columns

df = df.sample(n=60000, random_state=42).reset_index(drop=True)


df = df.dropna(subset=["article", "abstract"])
df = df[df["article"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
df = df[df["abstract"].apply(lambda x: isinstance(x, str) and x.strip() != "")]


from datasets import Dataset
dataset = Dataset.from_pandas(df)


print(dataset)


In [ ]:
train_dataset=dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template


tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

r1_prompt = """You are a reflective assistant engaging in thorough, iterative reasoning creating a summary of text, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>

<response>
{}
</response>
"""

EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    inputs = examples["article"]
    outputs = examples["abstract"]
    texts = []

    for inp, out in zip(inputs, outputs):

        text = r1_prompt.format(inp, out) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

print("Sample formatted text:\n")
print(train_dataset[0]["text"][:1000]) 

train_dataset.save_to_disk("pubmed_llama3_formatted")
print(" Dataset formatted to pubmed_llama3_formatted")


In [ ]:
from datasets import load_from_disk, concatenate_datasets
from tqdm import tqdm
import math, os, gc
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length,
    )


CHUNK_SIZE = 2000  
os.makedirs("tokenized_chunks", exist_ok=True)
num_chunks = math.ceil(len(train_dataset) / CHUNK_SIZE)

for i in tqdm(range(num_chunks)):
    start, end = i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, len(train_dataset))
    chunk = train_dataset.select(range(start, end))

    tokenized_chunk = chunk.map(
        tokenize_function,
        batched=True,
        batch_size=100,
        num_proc=1,
        load_from_cache_file=False,
    )

    tokenized_chunk.save_to_disk(f"tokenized_chunks/chunk_{i}")

    del chunk
    del tokenized_chunk
    gc.collect()  
    os.system("sync")  
    os.system("echo 3 > /proc/sys/vm/drop_caches 2>/dev/null || true")  


chunks = [load_from_disk(f"tokenized_chunks/{p}") for p in sorted(os.listdir("tokenized_chunks"))]
tokenized_dataset = concatenate_datasets(chunks)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported



trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_dataset,
    dataset_text_field=None, 
    max_seq_length=max_seq_length, 
    dataset_num_proc=1,
    args=TrainingArguments(
        output_dir="outputs1",

        per_device_train_batch_size=2,  
        gradient_accumulation_steps=16, 
        
    
        num_train_epochs=1,            
        save_strategy="steps",
        save_steps=100, 
        save_total_limit=3, 
        warmup_steps=100, 
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25, 
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)

In [ ]:

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
import os


SAVE_DIR = "/kaggle/working/outputs1/final_model"
os.makedirs(SAVE_DIR, exist_ok=True)


model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f" Model and tokenizer saved successfully at: {SAVE_DIR}")

In [ ]:
#saving model
from unsloth import FastLanguageModel
from huggingface_hub import login
import os

hf_token = ""
login(token=hf_token)

MODEL_DIR = ""
HF_REPO_ID =""

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_DIR,
    max_seq_length=1024,
    dtype=None,       
    load_in_4bit=True 
)

SAVE_PATH = ""
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

from transformers import AutoModelForCausalLM, AutoTokenizer


model = AutoModelForCausalLM.from_pretrained(SAVE_PATH)
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)

print(f"Model pushed to https://huggingface.co/{HF_REPO_ID}")


In [ ]:
#loading model
from unsloth import FastLanguageModel
from transformers import TextStreamer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/working/outputs1/final_model",  
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)  


article = '''ARTICLE:
'''
prompt = f"Summarize the following article in 3-4 concise sentences:\n\n{article}"

messages = [
    {"role": "user", "content": prompt},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

streamer = TextStreamer(tokenizer, skip_prompt=True)
outputs = model.generate(
    input_ids = inputs,
    streamer = streamer,
    max_new_tokens = 200,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n🧠 Generated Summary:\n", summary)
